# Fase 0 — Pilot Crawler

**Tujuan:** memverifikasi struktur HTML DetikFinance (indeks + artikel), mengukur
volume berita & kebutuhan disk, dan menguji selektor di tiga era HTML (2022 / 2024 / 2026)
**sebelum** meng-commit ke crawl rentang penuh. Mengacu spec §2.5 & §5.1.

## Dua output crawl (dan mapping di antaranya)

Crawler menghasilkan **2 output**, dan keduanya ditautkan lewat `article_id`
(angka setelah `d-` pada URL, mis. `.../d-8688622/...` → id `8688622`):

1. **Tabel link (mapping)** — disimpan sebagai **Excel (.xlsx)** agar mudah kamu buka,
   dan **parquet** untuk dipakai tahap berikutnya. Kolom kunci:
   `article_id, url, title, subchannel, date_text, index_date, html_path`.
   Kolom `html_path` = jembatan eksplisit ke file HTML.
2. **HTML mentah (.gz)** — disimpan di `data/raw/html_article/{id[:3]}/{id}.html.gz`.
   Nama file = `article_id`, jadi dari link bisa langsung ketemu file HTML-nya, dan sebaliknya.

| Dari | Ke | Caranya |
|------|----|---------| 
| URL `.../d-8688622/...` | file HTML | `html_article/868/8688622.html.gz` |
| file HTML `8688622.html.gz` | URL asal | kolom `url` pada baris `article_id==8688622` di tabel mapping |

## Yang sudah terverifikasi

| Hal | Temuan |
|-----|--------|
| URL indeks | `?page={n}&date=MM/DD/YYYY` — server-side, tak perlu JS |
| URL artikel | `/{subchannel}/d-{id}/{slug}` |
| Item/halaman | ~20; hari normal ~6 halaman (~110 artikel/hari) |
| Waktu terbit | `<meta name=publishdate>` presisi detik, WIB |
| Judul/body | `h1.detail__title` / `.detail__body-text` — stabil 2022/24/26 |
| Disk gz | ~48 KB/artikel, ~25 KB/indeks → skenario C ≈ 1.6 GB |


In [ ]:
import sys, yaml
from datetime import date
from pathlib import Path
import pandas as pd

sys.path.insert(0, "..")
from src.crawl import (
    Crawler, CrawlConfig, save_html_gz,
    index_html_path, article_html_path,
    article_html_relpath, index_html_relpath,
)
from src.parse import parse_index, parse_article

cfg = yaml.safe_load(open("../config/config.yaml"))
sel = cfg["selectors"]
RAW = Path("../data/raw")
INTERIM = Path("../data/interim"); INTERIM.mkdir(parents=True, exist_ok=True)

## 1. Crawler sopan (produksi: `request_delay_sec=1.0`)

In [ ]:
cc = CrawlConfig(
    user_agent=cfg["crawl"]["user_agent"],  # GANTI kontak email di config.yaml
    request_delay_sec=0.3,                   # pilot; produksi = 1.0
    jitter_sec=(0.1, 0.3),
)
cw = Crawler(cfg=cc)

## 2. Crawl indeks → bangun TABEL LINK (output #1) + simpan HTML (output #2)

In [ ]:
sample_days = [date(2022, 3, 1), date(2024, 3, 1), date(2026, 3, 2)]

link_rows = []        # -> output #1: tabel link/mapping
idx_bytes = art_bytes = n_art = 0

for d in sample_days:
    # --- indeks: simpan HTML indeks + kumpulkan semua link ---
    html = cw.fetch_index_page(d, page=1)
    idx_bytes += save_html_gz(html, index_html_path(RAW, d, 1))
    items = parse_index(html, sel)
    print(f"{d}: {len(items)} items on page 1")
    for it in items:
        link_rows.append({
            "article_id":  it.article_id,
            "url":         it.url,
            "title":       it.title,
            "subchannel":  it.subchannel,
            "date_text":   it.date_text,
            "index_date":  d.isoformat(),
            "html_path":   article_html_relpath(it.article_id),  # jembatan ke file HTML
            "html_saved":  False,                                 # diisi True saat HTML diunduh
        })

links = pd.DataFrame(link_rows)
print(f"\nTotal link terkumpul (page 1 saja): {len(links)}")
links.head()

In [ ]:
# --- unduh HTML untuk sebagian artikel (pilot) & tandai html_saved=True ---
sample_ids = links["article_id"].head(9).tolist()
for aid, url in links.loc[links.article_id.isin(sample_ids), ["article_id", "url"]].values:
    ahtml = cw.fetch_article(url)
    art_bytes += save_html_gz(ahtml, article_html_path(RAW, aid))
    links.loc[links.article_id == aid, "html_saved"] = True
    n_art += 1

print(f"HTML artikel diunduh: {n_art}")
links[links.html_saved].head()

## 3. SIMPAN output #1 ke Excel + parquet

- **`.xlsx`** → untuk kamu buka & akses langsung.
- **`.parquet`** → dipakai tahap berikutnya (cepat, hemat untuk puluhan ribu baris).

In [ ]:
xlsx_path = INTERIM / "news_links_pilot.xlsx"
pq_path   = INTERIM / "news_links_pilot.parquet"

links.to_excel(xlsx_path, index=False)       # butuh openpyxl
links.to_parquet(pq_path, index=False)
print("Excel  :", xlsx_path.resolve())
print("Parquet:", pq_path.resolve())
print("\nBuka news_links_pilot.xlsx -> tiap baris = 1 link + kolom html_path ke file HTML-nya.")

## 4. Buktikan mapping link <-> HTML dua arah

In [ ]:
from src.crawl import load_html_gz

row = links[links.html_saved].iloc[0]
print("LINK  :", row.url)
print("ID    :", row.article_id)
print("HTML  :", row.html_path)

# link -> HTML -> parse balik -> cocokkan judul
html = load_html_gz(RAW / row.html_path)
art  = parse_article(html, row.url, sel)
print("\nParse balik dari file HTML:")
print("  judul    :", art.title[:70])
print("  published:", art.published_at)
print("  n_words  :", art.n_words)

In [ ]:
print(f"index   : ~{idx_bytes/3/1024:.1f} KB/page (gz)")
print(f"article : ~{art_bytes/n_art/1024:.1f} KB/article (gz)")
proj = (idx_bytes/3*17000 + art_bytes/n_art*25000)/(1024**3)
print(f"PROYEKSI skenario C (gz): ~{proj:.2f} GB total")

## 5. (opsional) Kedalaman paginasi satu hari penuh

In [ ]:
def crawl_full_day(day):
    out = []
    for page in range(1, 50):
        html = cw.fetch_index_page(day, page)
        items = parse_index(html, sel)
        if not items:
            break
        out.extend(items)
    return out

items = crawl_full_day(date(2024, 3, 1))
print(f"Total artikel 2024-03-01: {len(items)}")
pd.Series([i.subchannel for i in items]).value_counts().head(12)